In [1]:
!pip3 install bitsandbytes==0.43.3
!pip install -U "huggingface_hub[cli]"
!pip install --upgrade transformers
!pip install -U bitsandbytes>=0.46.1
!pip install trl
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 19.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 31.1 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 27.0 MB/s eta 0:00:00


In [2]:
# from huggingface_hub import login

# login()  # will prompt in the cell

In [2]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
import torch
from datasets import load_dataset

import peft
from peft import LoraConfig, AutoPeftModelForCausalLM, get_peft_model

from trl import SFTTrainer

import os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

use_quantization = device.type == "cuda"

if use_quantization:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    quantized_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

test_question = (
    "Natalia sold clips to 48 of her friends in April, and then she sold half as "
    "many clips in May. How many clips did Natalia sell altogether in April and May?"
)

messages = [{"role": "user", "content": test_question}]
formatted_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
input_len = inputs["input_ids"].shape[1]

response = quantized_model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(response[0][input_len:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In April, Natalia sold 48 clips.
In May, she sold half as many clips as she sold in April, so she sold 48/2 = 24 clips.
Therefore, Natalia sold a total of 48 + 24 = 72 clips in April and May.
#### 72
The answer is: 72


In [4]:
print(quantized_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=960, out_features=960, bias=False)
          (k_proj): Linear8bitLt(in_features=960, out_features=320, bias=False)
          (v_proj): Linear8bitLt(in_features=960, out_features=320, bias=False)
          (o_proj): Linear8bitLt(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear8bitLt(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear8bitLt(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (nor

In [8]:
dataset = "openai/gsm8k"
data = load_dataset(dataset, 'main')

tokenizer.pad_token = tokenizer.eos_token
data = data.map(lambda samples: tokenizer(samples["question"], samples["answer"], truncation=True, padding="max_length", max_length=100), batched=True)

train_sample = data["train"].select(range(2000))

# display(train_sample)
display(data)



Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask'],
        num_rows: 1319
    })
})

In [6]:
print(train_sample[:1])

{'question': ['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'], 'answer': ['Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'], 'input_ids': [[62, 6927, 542, 3459, 23026, 288, 216, 36, 40, 282, 874, 2428, 281, 4124, 28, 284, 965, 1041, 3459, 2745, 347, 800, 23026, 281, 2405, 30, 1073, 800, 23026, 1250, 36366, 542, 5948, 13587, 281, 4124, 284, 2405, 47, 62, 6927, 542, 3459, 216, 36, 40, 31, 34, 446, 22646, 36, 40, 31, 34, 45, 34, 36, 7791, 34, 36, 23026, 281, 2405, 30, 198, 62, 6927, 542, 3459, 216, 36, 40, 27, 34, 36, 446, 22646, 36, 40, 27, 34, 36, 45, 39, 34, 7791, 39, 34, 23026, 13587, 281, 4124, 284, 2405, 30, 198, 1229, 216, 39, 34]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [7]:
# for name, module in quantized_model.named_modules():
#     print(name)

In [ ]:
# setup the lora Config
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)




# Training Arguments
working_dir = './'
output_directory = os.path.join(working_dir, "lora")

training_args = TrainingArguments(
    output_dir = output_directory,
    auto_find_batch_size = True,
    learning_rate = 3e-4,
    num_train_epochs=10
)


# Setting up the Trainer

trainer = SFTTrainer(
    model = quantized_model,
    args = training_args,
    train_dataset = train_sample,
    peft_config = lora_config,
    processing_class = tokenizer,
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# traii
trainer.train()




Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
500,1.534703
1000,1.421633
1500,1.392794
2000,1.378680
2500,1.373114


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 duri

TrainOutput(global_step=2500, training_loss=1.4201846923828125, metrics={'train_runtime': 777.8026, 'train_samples_per_second': 25.713, 'train_steps_per_second': 3.214, 'total_flos': 3787418880000000.0, 'train_loss': 1.4201846923828125})

In [9]:
# Save the model
trainer.save_model(output_directory)

In [10]:
print(output_directory)

./lora


In [11]:
model_path = output_directory

if device.type == "cuda":
    loaded_model = AutoPeftModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype="auto",
    )
else:
    loaded_model = AutoPeftModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype="auto",
    ).to(device)

loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

inputs = loaded_tokenizer(
    "Natalia sold clips to 48 friends in April, and half as many in May. How many clips in total?",
    return_tensors="pt",
).to(loaded_model.device)

outputs = loaded_model.generate(**inputs, max_new_tokens=100)

print(loaded_tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Natalia sold clips to 48 friends in April, and half as many in May. How many clips in total?In April, Natalia sold 48/2 = <<48/2=24>>24 clips.
In May, Natalia sold 24/2 = <<24/2=12>>12 clips.
In total, Natalia sold 24+12 = <<24+12=36>>36 clips.
#### 36
The answer is: 36
The cost of 1 clip is


In [12]:
test_questions = [
    "Natalia sold clips to 48 friends in April, and half as many in May. How many clips in total?",
    "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?",
    "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?"
]

print(f"Comparing Base Model vs Fine-Tuned Model ({model_name})\n")

for q in test_questions:
    print("=" * 80)
    print(f"Question: {q}")

    # Format input using the chat template so the instruct model responds properly
    messages = [{"role": "user", "content": q}]
    formatted_prompt = loaded_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = loaded_tokenizer(formatted_prompt, return_tensors="pt").to(loaded_model.device)
    input_len = inputs["input_ids"].shape[1]

    # Base model response (LoRA adapter disabled)
    with loaded_model.disable_adapter():
        outputs_base = loaded_model.generate(**inputs, max_new_tokens=100, pad_token_id=loaded_tokenizer.eos_token_id)
        answer_base = loaded_tokenizer.decode(outputs_base[0][input_len:], skip_special_tokens=True).strip()

    # Fine-tuned response (LoRA adapter active)
    outputs_ft = loaded_model.generate(**inputs, max_new_tokens=100, pad_token_id=loaded_tokenizer.eos_token_id)
    answer_ft = loaded_tokenizer.decode(outputs_ft[0][input_len:], skip_special_tokens=True).strip()

    print(f"\n[Base Model Output]:\n{answer_base if answer_base else '(no output generated)'}")
    print(f"\n[Fine-Tuned Output]:\n{answer_ft if answer_ft else '(no output generated)'}")
    print("=" * 80 + "\n")

Comparing Base Model vs Fine-Tuned Model (HuggingFaceTB/SmolLM2-360M-Instruct)

Question: Natalia sold clips to 48 friends in April, and half as many in May. How many clips in total?

[Base Model Output]:
In April, Natalia sold 48 friends, so she sold 48 / 2 = 24 clips.
In May, Natalia sold half as many friends, so she sold 24 / 2 = 12 friends.
Therefore, Natalia sold 24 + 12 = 36 clips in total.
#### 36
The answer is: 36

[Fine-Tuned Output]:
If Natalia sold 48 friends in April, then she sold 48/2 = <<48/2=24>>24 friends in May.
So, Natalia sold a total of 48 + 24 = <<48+24=72>>72 clips in April and May.
#### 72
The total number of clips sold is 72.
#### 72
The answer is 72.

Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

[Base Model Output]:
Yesterday, Weng did 50 minutes of babysitting, which is equivalent to 50/60 hours. Since she earns $12 an hour, she earned $12 * 50/60 = $12 * 1/2 = $6.

[Fine-Tuned Ou